# AI-Based E-Commerce Fraud Detection: Exploratory Data Analysis & Modeling

**Author**: Senior Data Scientist / ML Engineer  
**Project**: AI-Based E-Commerce Fraud Detection System  
**Objective**: Rigorous exploratory data analysis (EDA), anomaly profiling, class-imbalance investigation, feature engineering, and model diagnostics for credit card transaction streams.

---

## 1. Environment Setup & Library Imports

In [ ]:
import sys
from pathlib import Path

# Add project root to Python module search path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import RAW_DATA_PATH, BENCHMARK_DATA_PATH, FIGURES_DIR, TARGET_COL, AMOUNT_COL, TIME_COL
from src.data_loader import load_data

# Set high-resolution plotting aesthetics
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 150
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

print("Project Root:", project_root)
print("Figures Directory:", FIGURES_DIR)

## 2. Dataset Ingestion & Overview

In [ ]:
# Load dataset (loads official raw dataset if present, or generated benchmark)
df = load_data()
print("Dataset Shape:", df.shape)
df.head()

In [ ]:
# Data types and non-null counts
df.info()

In [ ]:
# Check for missing and duplicate values
missing_vals = df.isnull().sum().sum()
duplicates = df.duplicated().sum()
print(f"Total Missing Values: {missing_vals}")
print(f"Total Duplicate Rows: {duplicates}")

## 3. Fraud vs Genuine Class Distribution Analysis

In [ ]:
fraud_counts = df[TARGET_COL].value_counts()
fraud_pct = df[TARGET_COL].value_counts(normalize=True) * 100

summary_df = pd.DataFrame({
    "Transaction Count": fraud_counts,
    "Percentage (%)": fraud_pct
})
summary_df.index = ["Genuine (0)", "Fraud (1)"]
display(summary_df)

# Visualization: Class Distribution
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
sns.countplot(data=df, x=TARGET_COL, palette=["#3B82F6", "#EF4444"], ax=ax[0])
ax[0].set_title("Class Distribution (Absolute Counts)", weight="bold")
ax[0].set_xticklabels(["Genuine", "Fraud"])
ax[0].set_yscale("log")
ax[0].set_ylabel("Count (Log Scale)")

ax[1].pie(fraud_counts, labels=["Genuine", "Fraud"], autopct="%1.2f%%", colors=["#3B82F6", "#EF4444"], explode=[0, 0.15], startangle=140)
ax[1].set_title("Fraud Ratio in Transactions", weight="bold")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "class_distribution.png", dpi=300)
plt.show()

## 4. Transaction Amount Statistics & Distribution

In [ ]:
print("=== Genuine Transaction Amount Stats ===")
display(df[df[TARGET_COL] == 0][AMOUNT_COL].describe())

print("=== Fraudulent Transaction Amount Stats ===")
display(df[df[TARGET_COL] == 1][AMOUNT_COL].describe())

# Amount Distribution Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.histplot(df[df[TARGET_COL] == 0][AMOUNT_COL], bins=50, color="#3B82F6", kde=True, ax=axes[0], label="Genuine")
sns.histplot(df[df[TARGET_COL] == 1][AMOUNT_COL], bins=50, color="#EF4444", kde=True, ax=axes[0], label="Fraud")
axes[0].set_title("Transaction Amount Distribution (Linear Scale)", weight="bold")
axes[0].set_xlim([0, 1500])
axes[0].legend()

sns.histplot(np.log1p(df[df[TARGET_COL] == 0][AMOUNT_COL]), bins=50, color="#3B82F6", kde=True, ax=axes[1], label="Genuine")
sns.histplot(np.log1p(df[df[TARGET_COL] == 1][AMOUNT_COL]), bins=50, color="#EF4444", kde=True, ax=axes[1], label="Fraud")
axes[1].set_title("Log-Transformed Amount Distribution", weight="bold")
axes[1].set_xlabel("log1p(Amount)")
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / "amount_distribution.png", dpi=300)
plt.show()

## 5. Temporal Transaction Rhythm

In [ ]:
df["hour_of_day"] = (df[TIME_COL] / 3600.0) % 24.0

plt.figure(figsize=(10, 4.5))
sns.kdeplot(df[df[TARGET_COL] == 0]["hour_of_day"], color="#3B82F6", label="Genuine Transactions", fill=True, alpha=0.3)
sns.kdeplot(df[df[TARGET_COL] == 1]["hour_of_day"], color="#EF4444", label="Fraud Transactions", fill=True, alpha=0.3)
plt.title("Transaction Density by Hour of the Day (48-Hour Cycle)", weight="bold")
plt.xlabel("Hour of Day (0 - 23)")
plt.ylabel("Density")
plt.xlim([0, 24])
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "time_distribution.png", dpi=300)
plt.show()

## 6. Correlation Analysis of PCA Features with Fraud Target

In [ ]:
# Compute correlation of all numerical features with the target variable
correlations = df.corr()[TARGET_COL].drop(TARGET_COL).sort_values()

plt.figure(figsize=(12, 5))
colors = ["#EF4444" if x < 0 else "#3B82F6" for x in correlations]
correlations.plot(kind="bar", color=colors)
plt.title("Feature Correlation with Fraud Label (Class)", weight="bold", fontsize=13)
plt.ylabel("Pearson Correlation Coefficient")
plt.xlabel("Features")
plt.axhline(0, color="black", lw=0.8, linestyle="--")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "correlation_heatmap.png", dpi=300)
plt.show()

print("Top Negative Correlating Features:")
print(correlations.head(5))
print("\nTop Positive Correlating Features:")
print(correlations.tail(5))

## 7. Latent Anomaly Distributions (V14, V12, V17, V4)

In [ ]:
key_features = ["V14", "V12", "V17", "V4"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    if feat in df.columns:
        sns.boxplot(data=df, x=TARGET_COL, y=feat, palette=["#3B82F6", "#EF4444"], ax=axes[i])
        axes[i].set_title(f"Distribution of {feat} by Class", weight="bold")
        axes[i].set_xticklabels(["Genuine (0)", "Fraud (1)"])

plt.tight_layout()
plt.savefig(FIGURES_DIR / "feature_distributions.png", dpi=300)
plt.show()

## 8. Summary of Findings & Next Steps

1. **Severe Imbalance**: Fraud accounts for ~0.17% of total volume. Models must be evaluated on PR-AUC, F1-Score, and Recall rather than raw accuracy.
2. **Feature Discriminability**: Features `V14`, `V12`, and `V17` exhibit substantial negative divergence in fraudulent transactions, while `V4` and `V11` show strong positive elevation.
3. **Amount Skew**: Amount exhibits an extreme right-skewed heavy tail; logarithmic transformation (`log1p`) stabilizes variance.
4. **Pipeline Strategy**: Stratified splitting, SMOTE on training data only, and tree-based gradient boosting (XGBoost) provide robust fraud detection with calibrated probability scores.